# Riesgo y predicción cuantitativa: factores asociados al mal riesgo crediticio

**Curso:** MCC002 - Probabilidad y Estadística Computacional 

**Grupo 5:** 
- Armando Castro Chaupis 
- Henry Sánchez Alvarado 
- Alex Segura Núñez

**Pregunta principal:** ¿Qué factores explican la probabilidad de que un solicitante sea clasificado como mal riesgo crediticio?

**Preguntas secundarias:**

1. Proporción global de malos créditos y su IC 95 %.
2. ¿La tasa de mal crédito difiere según el propósito del préstamo?
3. ¿La duración y el monto del crédito difieren entre buenos y malos créditos?
4. ¿Qué variables están asociadas con mayor riesgo crediticio?
5. ¿Cómo cambia la clasificación al modificar el umbral de decisión?

## 1. Importación de librerias

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn
from scipy import stats 

### Establecemos valor de semilla
SEED = 2026
np.random.seed(SEED)

### Estilo de gráficos para matplotlib
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12


# 2. Carga de dataset

import os, io, zipfile, urllib.request

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"
CANDIDATOS = ["data/german.data", "german.data"]

ruta_datos = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if ruta_datos is None:
    # Descarga documentada desde la fuente oficial (solo si no existe copia local)
    os.makedirs("data", exist_ok=True)
    print("Descargando dataset desde UCI...")
    with urllib.request.urlopen(UCI_ZIP_URL) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extract("german.data", path="data")
    ruta_datos = "data/german.data"

# Nombres de columnas según german.doc (atributos 1..20 + clase)
columnas = [
    "estado_cuenta", "duracion", "historial_credito", "proposito", "monto",
    "ahorros", "empleo_actual", "tasa_cuota", "estatus_personal_sexo",
    "otros_deudores", "anios_residencia", "propiedad", "edad",
    "otros_planes_pago", "vivienda", "n_creditos_banco", "trabajo",
    "n_dependientes", "telefono", "trabajador_extranjero", "clase",
]

df = pd.read_csv(ruta_datos, sep=" ", header=None, names=columnas)
print(f"Archivo cargado: {ruta_datos}")
print(f"Dimensiones: {df.shape[0]} observaciones x {df.shape[1]} columnas")
df.head()

### 3.1 Diccionario de variables

Construido a partir del dataset. Las 13 variables cualitativas usan códigos `A**`. 
Abajo se documenta el significado de cada código y se crea un mapeo de etiquetas.

| # | Variable (nombre en el notebook) | Tipo | Descripción |
|---|---|---|---|
| 1 | `estado_cuenta` | Categórica ordinal | Estado de la cuenta corriente: A11 (< 0 DM), A12 (0–200 DM), A13 (≥ 200 DM), A14 (sin cuenta) |
| 2 | `duracion` | Numérica (meses) | Duración del crédito |
| 3 | `historial_credito` | Categórica | A30–A34: de "sin créditos/todo pagado" a "cuenta crítica/créditos en otros bancos" |
| 4 | `proposito` | Categórica nominal | A40–A410: auto nuevo/usado, mobiliario, radio/TV, electrodomésticos, reparaciones, educación, reentrenamiento, negocio, otros |
| 5 | `monto` | Numérica (DM) | Monto del crédito |
| 6 | `ahorros` | Categórica ordinal | A61–A65: nivel de ahorros/bonos (A65 = desconocido/sin cuenta) |
| 7 | `empleo_actual` | Categórica ordinal | A71–A75: antigüedad en el empleo actual |
| 8 | `tasa_cuota` | Numérica (1–4) | Cuota como % del ingreso disponible |
| 9 | `estatus_personal_sexo` | Categórica | A91–A94: estado civil y sexo (composición histórica del dataset) |
| 10 | `otros_deudores` | Categórica | A101 ninguno, A102 co-solicitante, A103 garante |
| 11 | `anios_residencia` | Numérica (1–4) | Años en la residencia actual |
| 12 | `propiedad` | Categórica | A121 inmueble … A124 sin propiedad conocida |
| 13 | `edad` | Numérica (años) | Edad del solicitante |
| 14 | `otros_planes_pago` | Categórica | A141 banco, A142 tiendas, A143 ninguno |
| 15 | `vivienda` | Categórica | A151 alquilada, A152 propia, A153 gratuita |
| 16 | `n_creditos_banco` | Numérica | N.º de créditos existentes en este banco |
| 17 | `trabajo` | Categórica ordinal | A171–A174: de no calificado/no residente a directivo/independiente |
| 18 | `n_dependientes` | Numérica | Personas a cargo |
| 19 | `telefono` | Binaria | A191 no, A192 sí (registrado) |
| 20 | `trabajador_extranjero` | Binaria | A201 sí, A202 no |
| 21 | `clase` | **Variable respuesta** | 1 = buen riesgo, 2 = mal riesgo |

In [ ]:
# Mapeos de códigos Axy -> etiquetas legibles (según german.doc)
mapa_proposito = {
    "A40": "Auto nuevo", "A41": "Auto usado", "A42": "Mobiliario/equipos",
    "A43": "Radio/TV", "A44": "Electrodomésticos", "A45": "Reparaciones",
    "A46": "Educación", "A48": "Reentrenamiento", "A49": "Negocio", "A410": "Otros",
}
mapa_estado_cuenta = {
    "A11": "< 0 DM", "A12": "0 – 200 DM", "A13": "≥ 200 DM", "A14": "Sin cuenta",
}
mapa_ahorros = {
    "A61": "< 100 DM", "A62": "100 – 500 DM", "A63": "500 – 1000 DM",
    "A64": "≥ 1000 DM", "A65": "Desconocido/sin cuenta",
}
mapa_historial = {
    "A30": "Sin créditos/todo pagado", "A31": "Pagados en este banco",
    "A32": "Al día hasta ahora", "A33": "Retrasos en el pasado",
    "A34": "Cuenta crítica/otros bancos",
}

df["proposito_lbl"] = df["proposito"].map(mapa_proposito)
df["estado_cuenta_lbl"] = df["estado_cuenta"].map(mapa_estado_cuenta)

# Verificación de que no quedaron códigos sin mapear
assert df["proposito_lbl"].notna().all(), "Hay códigos de propósito sin mapear"
assert df["estado_cuenta_lbl"].notna().all(), "Hay códigos de estado de cuenta sin mapear"

vars_numericas = ["duracion", "monto", "tasa_cuota", "anios_residencia",
                  "edad", "n_creditos_banco", "n_dependientes"]
vars_categoricas = [c for c in columnas if c not in vars_numericas + ["clase"]]
print(f"Variables numéricas ({len(vars_numericas)}): {vars_numericas}")
print(f"Variables categóricas ({len(vars_categoricas)}): {vars_categoricas}")

## Parte 2

## 9. Inferencia a través de la frecuencia
### 9.1 ¿La tasa de mal crédito difiere según el propósito?

Deberíamos esperar que para cada propósito exista la misma proporción de créditos de `riesgo` y `no-riesgo` (lo esperado, $E_i$) respecto a la proporción de toda la muestra. Sin embargo, en la práctica esto no siempre ocurre (lo observado, $O_i$). Para evaluar si la tasa de crédito difiere según el propósito, utilizaremos la prueba de **chi-cuadrado** $\chi^2$. Esta prueba se usa debido a la naturaleza categorica de las variables. 

$$
\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}
$$

Recordar que los grados de libertad de la distribucion se calculan $g_{dl}=(r-1)(c-1)$. Donde `r` es el número de propositos y `c` es la cantidad de clases. Para declarar que es válido la prueba de $\chi^2$, se debe cumplir los siguientes puntos:
- No debe existir frecuencias esperadas menor a 1.
- Como máximo, el 20% de las frecuencias esperadas deben ser menores a 5.

Despues de satisfacer esto, esto es posible hallar el valor `p` que nos permite aceptar o rechazar la **hipotesis de independencia** al 5\%. 


En caso se rechace la hipotesis, ¿que tan fuerte es esta dependencia entre variables? Para ello usarémos  **V de Cramér** como tamaño de efecto:
$$V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1, c-1)}}$$

Y se puede interpretar de la siguiente forma:

 | V          | Interpretación    |
| ---------- | ----------------- |
| 0          | Sin relación      |
| 0.10       | Relación pequeña  |
| 0.30       | Relación moderada |
| 0.50 o más | Relación fuerte   |



In [ ]:
## Codigo tal cual

tabla = pd.crosstab(df["proposito_lbl"], df["mal_credito"])
chi2, p_chi, dof, esperadas = stats.chi2_contingency(tabla)
V = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
n_esp_bajas = int((esperadas < 5).sum())

print("--- Prueba chi-cuadrado: clase x propósito (10 categorías) ---")
print(f"chi2 = {chi2:.2f}, gl = {dof}, p = {p_chi:.5f}, V de Cramér = {V:.3f}")
print(f"Celdas con frecuencia esperada < 5: {n_esp_bajas} de {tabla.size}")

In [ ]:
## Permite ver que tipo de variable tiene baja frecuencia esperada.
esp_df = pd.DataFrame(
    esperadas,
    index=tabla.index,
    columns=tabla.columns
)
 
for fila in esp_df.index:
    for col in esp_df.columns:
        if esp_df.loc[fila, col] < 5:
            print(f"{fila} - {col}: {esp_df.loc[fila, col]:.2f}")

A fin de mejorar la robustez de la estimación por la prueba del 

In [ ]:
frec = df["proposito_lbl"].value_counts()
raros = frec[frec < 30].index.tolist()
df["proposito_grp"] = df["proposito_lbl"].where(~df["proposito_lbl"].isin(raros),
                                                "Otros (agrupado)")
tabla_g = pd.crosstab(df["proposito_grp"], df["mal_credito"])
chi2_g, p_g, dof_g, esp_g = stats.chi2_contingency(tabla_g)
V_g = np.sqrt(chi2_g / (n * (min(tabla_g.shape) - 1)))
print(f"\n--- Robustez con categorías raras agrupadas ({tabla_g.shape[0]} categorías) ---")
print(f"Agrupados: {raros}")
print(f"chi2 = {chi2_g:.2f}, gl = {dof_g}, p = {p_g:.5f}, V de Cramér = {V_g:.3f}, "
      f"esperadas < 5: {int((esp_g < 5).sum())}")

De ambos procedimientos, se puede observar que la prueba de Chi-cuadrado muestra dependencia estadistica significativa entre el proposito y el riesgo del credito. Sin embargo, es importante resaltar que el tamaño de efecto `V` es pequeño, indicando que su relación es pequeña.

### 9.2 ¿Difieren la duración y el monto entre buenos y malos créditos?

### 9.2.1 Analisis de la variable duración
Debido a que son variables numericas, realizar una prueba de Chi-cuadrado no serviria. Por lo que, para la categoría **duración**: aplicamos la prueba **t de Welch**. Esta prueba no asume varianzas iguales como la **t de Student**. 

Al igual que en la sección anterior, para estimar que tan fuerte es esta dependencia usamos la `d de Cohen`, definido como:

$$
d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}
$$

donde:

- $\bar{x}_1$: media del grupo 1.
- $\bar{x}_2$: media del grupo 2.
- $s_{\text{pooled}}$: desviación estándar agrupada, calculada como:

$$
s_{\text{pooled}} =
\sqrt{
\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}
{n_1+n_2-2}
}
$$

A fin de interpretar el valor de `d`, se usa la siguiente convección:
- $d \approx 0.2$: efecto pequeño.
- $d \approx 0.5$: efecto mediano.
- $d \ge 0.8$: efecto grande.


In [ ]:
## Selección de buenos y malos creditos
buenos = df[df["mal_credito"] == 0]
malos  = df[df["mal_credito"] == 1]

## Definición de la d cohen

def d_cohen(a, b):
    s_pool = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / s_pool

t_w, p_w = stats.ttest_ind(malos["duracion"], buenos["duracion"], equal_var=False)
d_dur = d_cohen(malos["duracion"], buenos["duracion"])
print("--- Duración (meses): Welch ---")
print(f"media buenos = {buenos['duracion'].mean():.2f} | media malos = {malos['duracion'].mean():.2f}")
print(f"t = {t_w:.3f}, p = {p_w:.2e}, d de Cohen = {d_dur:.3f}")

Con el valor de `p`$<2.4e-10$ se rechaza la hipotesis de igualdad de medias. En la muestra, se observa que en promedio, los créditos malos presentan una duración mayor (24.86 meses) que los créditos buenos (19.21 meses). El tamaño del efecto fue `d`=0.480, correspondiente a un efecto moderado.

### 9.2.2 Analisis de la variable monto 
Para la variable `monto`, se observa que los valores están distribuidos en un rango mucho más amplio que los valores de la variable `duración`. Con el test de `Mann–Whitney` se comparan los rangos, por lo que no es necesario realizar alguna transformación a la variable `monto`.Es común utilizar el tamaño del efecto **$r$**.

Se calcula como:

$$
r = 1 - \frac{2U}{n_1 n_2}
$$

donde:

- $U$: estadístico de la prueba de Mann–Whitney.
- $n_1$: tamaño del primer grupo.
- $n_2$: tamaño del segundo grupo.

El valor de $r$ suele interpretar de la siguiente manera:

| $r$ | Interpretación |
|:-----:|:---------------|
| 0.0 | Sin diferencia |
| 0.1 | Efecto pequeño |
| 0.3 | Efecto moderado |
| 0.5 o mayor | Efecto grande |


Adicionalmente, calculamos que tan fuerte es la dependencia usando el `d de Cohen`. Sin embargo, transformamos los valores a $log(monto)$ debido a la asimetria de los valores `monto`.

In [ ]:
U, p_u = stats.mannwhitneyu(malos["monto"], buenos["monto"], alternative="two-sided")
r_rb = 1 - 2*U/(len(malos)*len(buenos))       
d_logm = d_cohen(malos["log_monto"], buenos["log_monto"])
print("\n--- Monto (DM): Mann-Whitney ---")
print(f"mediana buenos = {buenos['monto'].median():.0f} | mediana malos = {malos['monto'].median():.0f}")
print(f"U = {U:.0f}, p = {p_u:.4f}, r biserial de rangos = {r_rb:.3f}, "
      f"d de Cohen sobre log(monto) = {d_logm:.3f}")

Se observa que los malos creditos tienen montos mayores ($2574$ vs. $2244$) pero el efecto es **pequeño** (d sobre log-monto ≈ 0.24). La dirección negativa de $r$ nos indica el orden del cálculo de las medianas, donde el valor absoluto nos indica una relación pequeña respecto a la calidad del credito.


Sin embargo, es de notar que las variables  `monto` y `duración` están relacionadas por el alto valor de correlación obtenida ($0.62$).

## 11. Regresión logistica

Modelamos  la clasificación de si un prestamo es riesgoso o no:
$$\Pr(Y_i = 1 \mid \mathbf x_i) = \sigma(\mathbf x_i^\top \boldsymbol\beta) = \frac{1}{1 + e^{-\mathbf x_i^\top \boldsymbol\beta}}$$
Cada $e^{\beta_j}$ es un **odds ratio (OR)**: el factor multiplicativo sobre las *odds* de mal crédito por unidad del predictor $j$. 

Para la regresión, usaremos las variables con el respaldo del EDA: estado de la cuenta corriente, duración, log(monto), historial crediticio, ahorros, antigüedad laboral, tasa de cuota, edad, vivienda y otros planes de pago.

**Pasos a seguir:** 
- Partición estratificada 70/30 entrenamiento/prueba (manteniendo constante las proporciones en el dataset de entrenamiento y prueba). 
- Se realizará AUC por validación cruzada estratificada de 5 folds como chequeo de estabilidad, asi como los reportes de metricas al dataset de prueba.

En regresión logística no existe un coeficiente de determinación \(R^2\) equivalente al de la regresión lineal. En su lugar, se utiliza el **Pseudo-\(R^2\) de McFadden**, que mide cuánto mejora el modelo ajustado respecto a un modelo nulo (que solo incluye el intercepto).

Se define como:

$$
R^2_{\text{McFadden}}
=
1
-
\frac{\log L_{\text{modelo}}}
{\log L_{\text{nulo}}}
$$

donde:

- $\log L_{\text{modelo}}$: log-verosimilitud del modelo ajustado.
- $\log L_{\text{nulo}}$: log-verosimilitud del modelo que únicamente incluye el intercepto.


Como referencia general, pueden utilizarse los siguientes criterios:

| Pseudo-$R^2$ | Interpretación |
|:--------------:|:---------------|
| $< 0.10$ | Ajuste débil |
| $0.10 - 0.20$ | Ajuste aceptable |
| $0.20 - 0.40$ | Buen ajuste |
| $> 0.40$ | Ajuste excelente (poco frecuente) |


In [ ]:
### Selección de variables.
from sklearn.model_selection import train_test_split

FORMULA_LOGIT = ("mal_credito ~ C(estado_cuenta) + duracion + log_monto"
                 " + C(historial_credito) + C(ahorros) + C(empleo_actual)"
                 " + tasa_cuota + edad + C(vivienda) + C(otros_planes_pago)")

df_train, df_test = train_test_split(df, test_size=0.30, random_state=SEED,
                                     stratify=df["mal_credito"])
print(f"Entrenamiento: {len(df_train)} obs. (tasa mal crédito = {df_train['mal_credito'].mean():.3f})")
print(f"Prueba       : {len(df_test)} obs. (tasa mal crédito = {df_test['mal_credito'].mean():.3f})")

modelo_logit = smf.logit(FORMULA_LOGIT, data=df_train).fit(disp=0)

### revisar el pseudo-r2
print(f"\nPseudo-R2 de McFadden (train): {modelo_logit.prsquared:.4f}")
print(f"Convergencia: {modelo_logit.mle_retvals['converged']}")

Se obtiene un Pseudo-$R^2$ de $0.2202$, lo que nos explica que es un buen ajuste para nuestro clasificador logistico.

### Interpretación del valor \(p\) en la regresión logística

En una regresión logística, cada coeficiente se evalúa mediante una **prueba de hipótesis** para determinar si la variable está asociada significativamente con la probabilidad del evento de interés.

Las hipótesis son:

$$
H_0:\ \beta_j = 0
$$

$$
H_1:\ \beta_j \neq 0
$$

donde $\beta_j$ es el coeficiente asociado a la variable $j$.

El **valor $p$** representa la probabilidad de observar un coeficiente tan extremo como el estimado suponiendo que la hipótesis nula sea verdadera

La decisión se basa en un nivel de significancia, generalmente $\alpha = 0.05$:

- Si $p < 0.05$, se **rechaza la hipótesis nula**, concluyendo que la variable presenta una asociación estadísticamente significativa con la respuesta.
- Si $p \ge 0.05$, **no se rechaza la hipótesis nula**, por lo que no existe evidencia suficiente para afirmar que la variable tenga un efecto sobre la respuesta.


El valor $p$ **no mide el tamaño del efecto**. Una variable puede ser estadísticamente significativa ($p < 0.05$) pero tener un efecto pequeño. Por ello, el valor $p$ debe interpretarse conjuntamente con el **Odds Ratio (OR)** y su **intervalo de confianza del 95%**. Donde la interpretación de este ultimo es la siguiente:

- Si el **IC del 95% no contiene el valor 1**, existe evidencia de que la asociación entre la variable y la respuesta es estadísticamente significativa al nivel del 5%.
- Si el **IC del 95% contiene el valor 1**, no existe evidencia suficiente para afirmar que la variable tenga un efecto significativo sobre la respuesta.

In [ ]:
or_tabla = pd.DataFrame({
    "OR": np.exp(modelo_logit.params),
    "IC 2.5%": np.exp(modelo_logit.conf_int()[0]),
    "IC 97.5%": np.exp(modelo_logit.conf_int()[1]),
    "p-valor": modelo_logit.pvalues,
}).drop(index="Intercept").sort_values("OR")
or_tabla.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
tab = or_tabla.sort_values("OR")
colores = ["#2c7fb8" if o < 1 else "#d95f02" for o in tab["OR"]]
ax.errorbar(tab["OR"], range(len(tab)),
            xerr=[tab["OR"] - tab["IC 2.5%"], tab["IC 97.5%"] - tab["OR"]],
            fmt="o", ecolor="gray", elinewidth=1.2, capsize=3,
            markerfacecolor="white", markeredgecolor="k", zorder=3)
ax.scatter(tab["OR"], range(len(tab)), c=colores, s=42, zorder=4)
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_yticks(range(len(tab))); ax.set_yticklabels(tab.index, fontsize=8.5)
ax.set_xscale("log")
ax.set_xlabel("Odds ratio (escala log) — OR > 1 aumenta el riesgo")
ax.set_title("Figura 7. Odds ratios del modelo logístico (IC 95 %)")
plt.tight_layout(); plt.show()

### Respuesta a la pregunta 4: 

Por ejemplo, para el estado de la cuenta corriente, la categoría **A14** (sin cuenta corriente) presenta un **OR ≈ 0.18** respecto a la categoría de  `saldo negativo`. Esto significa que, manteniendo constantes las demás variables, las *odds* de presentar un mal crédito son aproximadamente un **82% menores** que las del grupo de referencia. De forma similar, la categoría **A13** (saldo mayor o igual a 200 DM) presenta un **OR ≈ 0.31**, por lo que también se asocia con una reducción importante de las *odds* de `mal crédito`.

Algo parecido ocurre con los ahorros, donde los clientes con **ahorros mayores o iguales a 1000 DM (A64)** presentan un **OR ≈ 0.16**, mientras que aquellos cuyo nivel de ahorro es **desconocido (A65)** presentan un **OR ≈ 0.43**. En ambos casos las *odds* de mal crédito son menores que las de la categoría de referencia. También se obtiene un OR menor que 1 para los clientes con **vivienda propia (A152)** y para quienes tienen una **antigüedad laboral entre 4 y 7 años (A74)**.

En cambio, la **duración del crédito** presenta un **OR ≈ 1.04 por cada mes adicional**. Como este valor es mayor que 1, cada mes extra incrementa las *odds* de mal crédito aproximadamente un **4%**, manteniendo constantes las demás variables. Debido a que este efecto es multiplicativo, un año adicional de duración equivale aproximadamente a un incremento del **60%** en las *odds*.. De manera similar, una mayor **tasa de cuota** también incrementa las *odds* de mal crédito (OR ≈ 1.23).

Un resultado interesante es el del **monto del crédito**. En el análisis bivariado (Sección 9.2) se encontró que los créditos malos tenían montos mayores que los créditos buenos. Sin embargo, en la regresión logística el **logaritmo del monto** presenta un **OR ≈ 1.07** con un **valor $p \approx 0.72$**, por lo que deja de ser estadísticamente significativo. Esto indica que, una vez consideradas las demás variables del modelo, el monto ya no aporta información adicional para explicar el riesgo de mal crédito.

La razón es que el monto y la duración están correlacionados (\(\rho \approx 0.62\)). En general, los créditos de mayor monto también suelen tener plazos más largos. Como la duración ya está incluida en el modelo y explica una parte importante del riesgo, el efecto que inicialmente parecía atribuirse al monto queda explicado por dicha variable. 

Finalmente, el resultado obtenido para la categoría **A34** debido a que presenta un **OR ≈ 0.24**, lo que sugiere menores *odds* de mal crédito respecto a la categoría de referencia **A30**. 

### 11.1 Desempeño del clasificador: 

Se usarán las metricas ROC–AUC, matriz de confusión, precision y recall para evaluación del modelo.

$$
Recall = \frac{TP}{TP+FN}
$$
$$
Precision = \frac{TP}{TP+FP}
$$

`Falso negativo`=> Cliente malo aprobado.

`Falso positivo`=> Cliente bueno rechazado.

`ROC–AUC`: probabilidad de que el modelo asigne mayor score a un cliente malo que a uno bueno elegidos al azar; es independiente del umbral y del balance de clases.